# Chapter 6 — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Chapter 6 - Fine-tuning for classification** of *'Build a Large Language Model (From Scratch)'* book by Sebastian Raschka.

### 0. Chapter Objective
Fine-tune a pretrained GPT model for binary text classification by adapting the dataset, replacing the output head, and training the model to predict `spam` or `not spam`.

### 1. Adding our repo root 'build-llm-from-scratch-pytorch' to sys.path

In [1]:
from pathlib import Path
import sys

# Current folder:
# repository_root/chapter_04/exercises
# i.e. 
# import os  
# print(os.getcwd()) # prints: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch\chapter_04\exercises 
# Note: Python searches for chapter_03 (and all other needed imports) inside that folder and in the other locations listed in sys.path. but sys.path currently does not have the repo root
# the snippet below adds the repo root to sys.path

repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)
sys.path

Repository root: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch


['c:\\Users\\delmi\\Documents\\LEARNING\\Manning_Learning\\repos\\build-llm-from-scratch-pytorch',
 'C:\\Users\\delmi\\anaconda3\\python312.zip',
 'C:\\Users\\delmi\\anaconda3\\DLLs',
 'C:\\Users\\delmi\\anaconda3\\Lib',
 'C:\\Users\\delmi\\anaconda3',
 'c:\\Users\\delmi\\venvs\\llmbookvenv',
 '',
 'c:\\Users\\delmi\\venvs\\llmbookvenv\\Lib\\site-packages']

### 2. create_balanced_dataset() and random_split() functions

In [ ]:
import pandas as pd
# Downsampling the majority class 'ham' class (because 'spam' class has way less samples)
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)

    balanced_df = pd.concat( [ham_subset, df[df["Label"] == "spam"]] ) # stacking them vertically
    return balanced_df

# balanced_df = create_balanced_dataset(df)
# print(balanced_df["Label"].value_counts())

def random_split(df, train_frac, validation_frac):
    df = df.sample(frac = 1, random_state=123).reset_index(drop=True) # shuffling the df
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

# train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

**Purpose:**

- `create_balanced_dataset()` undersamples the majority `ham` class to match the
  number of `spam` examples.
- `random_split()` shuffles and divides the balanced data into training,
  validation, and test sets.

### 10. Q/As
- **Q: The dataset is divided into three parts: training, validation, and testing. What is the purpose of each?**   
The **training set** is used to train the model,   
the **validation set** is used to adjust hyperparameters and prevent overfitting, and   
the **testing set** is used to evaluate the model's performance on unseen data. 

- **Q: Why is it often sufficient to fine-tune only the last layers of a pretrained LLM for a new task?**   
**The lower layers** of a pretrained LLM typically capture general language structures and semantics, while **the upper layers** learn task-specific features.  
**Fine-tuning only the last layers (i.e. the upper layers)** allows for efficient adaptation to new tasks without disrupting the learned general language knowledge.

- **Q: Why is the last token in a sequence considered the most informative for classification tasks using a causal attention mask?**  
The causal attention mask restricts each token's attention to itself and preceding tokens.  
As a result, the last token accumulates information from all previous tokens, making it the most comprehensive representation of the input sequence. 

- **Q: What factors influence the choice of the number of epochs during fine tuning?**  
The number of epochs depends on the **dataset's complexity and the task's difficulty**. **Overfitting** may necessitate **reducing the number of epochs**, while **insufficient training** might require **increasing them**.
